# Comparación de detectores, objetos y relaciones

YOLOv8n y YOLO26n se comparan sobre las mismas 2.619 imágenes de SUN RGB-D. El AP se calcula desde un umbral mínimo de confianza y la precisión, exhaustividad y F1 se calculan con el umbral operativo. Después se mide el efecto de la evidencia de objetos sobre las consultas compatibles. Visual Genome se utiliza únicamente para contrastar la cobertura de las reglas geométricas 2D.

In [ ]:
from pathlib import Path
import sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'semantic_navigation_ws' / 'src').is_dir())
sys.path.insert(0, str(repo / 'experiments' / 'shared'))
import pandas as pd
from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from offline_benchmarks import (create_detector_figures, run_detector_benchmark,
                                write_results_summary)
ctx = bootstrap_offline()
print(f"Dispositivo: {ctx['device']}")
display(pd.DataFrame([{'detector': key, 'checkpoint': value}
                      for key, value in ctx['config']['models']['yolo']['variants'].items()]))

## Detección y recuperación informada por objetos

In [ ]:
results = run_detector_benchmark(ctx)
display(results['detector_summary'])
display(results['object_average_precision'])
display(results['object_retrieval_summary'])

## Relaciones espaciales

La precisión frente a Visual Genome no equivale a una tasa directa de error: el conjunto solo anota relaciones salientes, mientras que las reglas emiten relaciones geométricas exhaustivas. La exhaustividad mide la cobertura de lo anotado.

In [ ]:
display(results['relation_metrics'])
display(results['relation_examples'])

## Exportación reproducible y figuras

In [ ]:
from reproducibility import collect_manifest, save_manifest
results_root = resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'detector_comparison'
figures_root = results_root / 'figures'
results_root.mkdir(parents=True, exist_ok=True)
for name, frame in results.items():
    frame.to_csv(results_root / f'{name}.csv', index=False)
figure_paths = create_detector_figures(results, figures_root)
vlm_root = resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'vlm_comparison'
vlm_results = {name: pd.read_csv(vlm_root / f'{name}.csv')
               for name in ('cases', 'summary', 'paired_differences',
                            'threshold_diagnostics', 'model_costs')}
summary_path = write_results_summary(
    resolve_repo_path(ctx['repo_root'], ctx['config']['paths']['results_root']) / 'OFFLINE_RESULTS.md',
    vlm_results, results)
manifest = collect_manifest(ctx['config'], repo_dir=str(ctx['repo_root']), device=ctx['device'],
    extra={'notebook': '02_yolo_and_relations',
           'detectors': ctx['config']['models']['yolo']['variants'],
           'object_retrieval_encoder': ctx['config']['models']['siglip']['object_retrieval_variant'],
           'figures': figure_paths})
save_manifest(str(results_root / 'manifest.json'), manifest)
print(f'Resultados: {results_root}')
print(f'Resumen consolidado: {summary_path}')
print('Figuras generadas:')
for path in figure_paths:
    print(' -', path)

## Límite de interpretación

Este cuaderno no evalúa multivista, contaminación entre habitaciones, política espacial ni éxito de navegación. Tampoco puede estimar offline la aportación de las relaciones a la recuperación porque el banco de consultas disponible no contiene ground truth relacional independiente. Esos factores quedan reservados a simulación.